## Build Model

In this notebook, we will build a U-Net model for the segmentation, which is widely used in medical image segmentation tasks. U-Net is an Encoder-Decoder architecture designed for biomedical image segmentation.


The architecture of U-Net can be found in the original [paper](https://arxiv.org/abs/1505.04597).

---

### 1. Importing Libraries

We will start by importing the necessary libraries for building the model using PyTorch.

In [1]:
import torch
import torch.nn as nn

---
### 2. Defining the Convolutional Block

A **convolutional block** consists of two convolutional layers, each followed by a ReLU activation function. 

This block will be used multiple times throughout our U-Net architecture, both in the encoder and decoder sections.

In [2]:
def conv_block(in_channels, out_channels):
    """Two convolutional layers followed by ReLU activations."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
        nn.ReLU()
    )

---
### 3. U-Net Architecture

The U-Net architecture is based on an **Encoder-Decoder** structure with **skip connections** at various layers to improve the model's ability to generate precise segmentations.

- **Encoder:** The encoder progressively reduces the image size by downsampling.
- **Decoder:** The decoder reconstructs the segmented mask to match the input image size through upsampling.
- **Skip connections:** These allow the network to pass features from the encoder directly to the decoder, maintaining essential details during reconstruction.

In [3]:
class UNet(nn.Module):
    """A simplified UNet architecture using maxpool for downsampling and bilinear upsampling."""
    def __init__(self):
        super(UNet, self).__init__()

        # Encoder (downsampling) layers
        self.enc1 = conv_block(1, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)

        # Decoder (upsampling) layers
        self.dec2 = conv_block(256 + 128, 128)
        self.dec1 = conv_block(128 + 64, 64)

        # Final output layer
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

        # Maxpool for downsampling
        self.maxpool = nn.MaxPool2d(2)

        # Upsample for upsampling
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')


    def forward(self, x_in):
        # Encoder path
        x1_e = self.enc1(x_in)
        x1_m = self.maxpool(x1_e)

        x2_e = self.enc2(x1_m)
        x2_m = self.maxpool(x2_e)

        x3_e = self.enc3(x2_m)

        # Decoder path
        y2_u = self.upsample(x3_e)
        y2_c = torch.cat((y2_u, x2_e), dim=1)
        y2_d = self.dec2(y2_c)

        y1_u = self.upsample(y2_d)
        y1_c = torch.cat((y1_u, x1_e), dim=1)
        y1_d = self.dec1(y1_c)

        # Final output layer
        out = self.final_conv(y1_d)

        return torch.sigmoid(out)

---
### 4. Testing the Model

Now, we will test the U-Net model with an example input to verify its functionality.

In [4]:
model = UNet()

x = torch.randn(1, 1, 128, 128)  # Example input tensor (batch size, channels, height, width)
output = model(x)
print(output.shape)  # Expected output shape (1, 1, 128, 128)

torch.Size([1, 1, 128, 128])


---
### 5. Inspecting the Model Structure

To get a better understanding of the model's internal structure, we can print the model architecture and calculate the total number of parameters.

In [5]:
print(model)
print("Number of parameters: ", sum([param.numel() for param in model.parameters()]))

UNet(
  (enc1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (enc2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (enc3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (dec2): Sequential(
    (0): Conv2d(384, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (dec1): Sequential(
    (0): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3

---
### 6. Visualizing Layer Outputs

To make it easier to understand what's happening inside the network, we have modified the U-Net architecture to print the size of the tensors at each layer.

In [6]:
class UNetDebug(nn.Module):
    # (Implementation similar to the previous UNet class, but with print statements in the forward method)
    # Prints tensor shapes after each layer

    def __init__(self):
        super(UNetDebug, self).__init__()

        # Encoder (downsampling) layers
        self.enc1 = conv_block(1, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)

        # Decoder (upsampling) layers
        self.dec2 = conv_block(256 + 128, 128)
        self.dec1 = conv_block(128 + 64, 64)

        # Final output layer
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

        # Maxpool for downsampling
        self.maxpool = nn.MaxPool2d(2)

        # Upsample for upsampling
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')

    def forward(self, x_in):
        print(f"x_in shape: {x_in.shape}")

        # Encoder path
        x1_e = self.enc1(x_in)
        print(f"x1_e shape: {x1_e.shape}")
        x1_m = self.maxpool(x1_e)
        print(f"x1_m shape: {x1_m.shape}")

        x2_e = self.enc2(x1_m)
        print(f"x2_e shape: {x2_e.shape}")
        x2_m = self.maxpool(x2_e)
        print(f"x2_m shape: {x2_m.shape}")

        x3_e = self.enc3(x2_m)
        print(f"x3_e shape: {x3_e.shape}")

        # Decoder path
        y2_u = self.upsample(x3_e)
        print(f"y2_u shape: {y2_u.shape}")
        y2_c = torch.cat((y2_u, x2_e), dim=1)
        print(f"y2_c shape: {y2_c.shape}")
        y2_d = self.dec2(y2_c)
        print(f"y2_d shape: {y2_d.shape}")

        y1_u = self.upsample(y2_d)
        print(f"y1_u shape: {y1_u.shape}")
        y1_c = torch.cat((y1_u, x1_e), dim=1)
        print(f"y1_c shape: {y1_c.shape}")
        y1_d = self.dec1(y1_c)
        print(f"y1_d shape: {y1_d.shape}")

        # Final output layer
        out = self.final_conv(y1_d)
        print(f"out shape: {out.shape}")

        return torch.sigmoid(out)

In [7]:
model = UNetDebug()

x = torch.randn(1, 1, 128, 128)  # Example input tensor (batch size, channels, height, width)
output = model(x)

x_in shape: torch.Size([1, 1, 128, 128])
x1_e shape: torch.Size([1, 64, 128, 128])
x1_m shape: torch.Size([1, 64, 64, 64])
x2_e shape: torch.Size([1, 128, 64, 64])
x2_m shape: torch.Size([1, 128, 32, 32])
x3_e shape: torch.Size([1, 256, 32, 32])
y2_u shape: torch.Size([1, 256, 64, 64])
y2_c shape: torch.Size([1, 384, 64, 64])
y2_d shape: torch.Size([1, 128, 64, 64])
y1_u shape: torch.Size([1, 128, 128, 128])
y1_c shape: torch.Size([1, 192, 128, 128])
y1_d shape: torch.Size([1, 64, 128, 128])
out shape: torch.Size([1, 1, 128, 128])
